In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical

# ========= 0. 固定隨機種子 =========
np.random.seed(42)
tf.random.set_seed(42)

# ========= 1. 參數設定 =========
DATA_PATH = "/content/1_075_05_025_0 (2023_25).csv"  # 若在 Colab/本地，視情況改這行
SEQ_LEN = 2
DEBUG_SHOW_EXAMPLES = 5

# ========= 2. 讀取原始資料 =========
df = pd.read_csv(DATA_PATH)

print("✅ 原始資料筆數：", len(df))
print("欄位：", list(df.columns))

required_cols = ["movie_id", "start_date", "box_office"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"缺少必要欄位：{c}")

df = df.dropna(subset=["movie_id", "start_date", "box_office"]).reset_index(drop=True)

# 轉數字
df["box_office"] = pd.to_numeric(df["box_office"], errors="coerce")
df = df.dropna(subset=["box_office"]).reset_index(drop=True)

# 轉日期
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
df = df.dropna(subset=["start_date"]).reset_index(drop=True)

# ====== 欄位名自動偵測（避免你那種 w1_change vs bo_diff_w0_w1 不一致）======
prev1_col_candidates = ["box_office_prev_w1", "prev_box_office"]
prev2_col_candidates = ["box_office_prev_w2", "prev2_box_office"]
diff_col_candidates  = ["bo_diff_w0_w1", "box_office_prev_w1_change", "prev_box_office_change"]

def pick_first_existing(cands, cols):
    for c in cands:
        if c in cols:
            return c
    return None

prev1_col = pick_first_existing(prev1_col_candidates, df.columns)
prev2_col = pick_first_existing(prev2_col_candidates, df.columns)
diff_col  = pick_first_existing(diff_col_candidates,  df.columns)

# prev 系列如果有，先轉數字
for col in [prev1_col, prev2_col, diff_col]:
    if col is not None:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 先算每篇評論的字數（之後週聚合用 sum）
title_col   = pick_first_existing(["review_title", "title"], df.columns)
content_col = pick_first_existing(["review_content", "content"], df.columns)

df["content_length"] = df.get(content_col, "").astype(str).str.len() if content_col else 0
df["title_length"]   = df.get(title_col, "").astype(str).str.len() if title_col else 0

df = df.sort_values(["movie_id", "start_date"]).reset_index(drop=True)
print("📅 清理後資料時間範圍：", df["start_date"].min(), "→", df["start_date"].max())

# ========= 3. 週聚合 =========
agg_dict = {
    "box_office": "first",
}

if prev1_col is not None:
    agg_dict[prev1_col] = "first"
if prev2_col is not None:
    agg_dict[prev2_col] = "first"
if diff_col is not None:
    agg_dict[diff_col] = "first"

# 互動量
reply_col = pick_first_existing(["reply_count"], df.columns)
if reply_col is not None:
    agg_dict[reply_col] = "sum"

# 情緒欄位
emotion_col = pick_first_existing(["sentiment_score", "gemini_emotion_analyse", "emotion_analyse"], df.columns)
if emotion_col is not None:
    agg_dict[emotion_col] = "mean"

# 代表性文字：一週只保留第一筆
for col in [title_col, content_col, "url"]:
    if col is not None and col in df.columns:
        agg_dict[col] = "first"

# 正負反應（如果有）
pos_col = pick_first_existing(["positive_reply_count", "positive_reaction_count"], df.columns)
neg_col = pick_first_existing(["negative_reply_count", "negative_reaction_count"], df.columns)
for col in [pos_col, neg_col]:
    if col is not None:
        agg_dict[col] = "sum"

# 文字長度
agg_dict["content_length"] = "sum"
agg_dict["title_length"]   = "sum"

weekly = (
    df.groupby(["movie_id", "start_date"], as_index=False)
      .agg(agg_dict)
      .sort_values(["movie_id", "start_date"])
      .reset_index(drop=True)
)

print("📈 週聚合後筆數：", len(weekly))
print("🪪 電影數量：", weekly["movie_id"].nunique())
print("⏱ 週資料時間範圍：", weekly["start_date"].min(), "→", weekly["start_date"].max())

dup = weekly.duplicated(subset=["movie_id", "start_date"]).sum()
print("🔍 weekly 中同電影同週重複筆數（應為 0）：", dup)
if dup != 0:
    raise RuntimeError("weekly 出現同電影同週重複，週聚合有問題。")

# ========= 3.5 加入「一年第幾週」週期特徵（解決 52 vs 1 相鄰問題） =========
# ISO week（1~52/53），不同年可能有 53 週；跨年那週用 iso_year 更穩
iso = weekly["start_date"].dt.isocalendar()
weekly["week_of_year"] = iso.week.astype(int)   # 1~52/53
weekly["iso_year"] = iso.year.astype(int)

# 每個 iso_year 的週數（52 或 53）
weeks_in_year = (
    weekly.groupby("iso_year")["week_of_year"]
          .transform("max")
          .astype(int)
)

# cyclic encoding：讓 week=52 和 week=1 在特徵空間很接近
weekly["week_sin"] = np.sin(2 * np.pi * weekly["week_of_year"] / weeks_in_year)
weekly["week_cos"] = np.cos(2 * np.pi * weekly["week_of_year"] / weeks_in_year)

# ========= 4. 建立 y_class (PR50 / PR80) =========
y_box = weekly["box_office"].values.astype(float)
p50 = np.percentile(y_box, 50)
p80 = np.percentile(y_box, 80)

print("\n📌 PR50 / PR80 門檻（基於週票房）")
print(f"  Low  : < {p50:.0f}")
print(f"  Mid  : {p50:.0f} ~ {p80:.0f}")
print(f"  High : ≥ {p80:.0f}")

def label_class(v):
    if v < p50:
        return 0
    elif v < p80:
        return 1
    else:
        return 2

weekly["y_class"] = weekly["box_office"].apply(label_class)

print("\n🎯 三類分布（weekly）：")
unique, counts = np.unique(weekly["y_class"].values, return_counts=True)
for u, c in zip(unique, counts):
    print(["Low", "Mid", "High"][u], f"({u}) :", c)

# ========= 5. 準備特徵欄位 =========
safe_feature_cols = [
    "box_office",          # t-2, t-1 的 box_office
    diff_col,
    reply_col,
    emotion_col,
    "content_length",
    "title_length",
    "week_sin",            # ✅ 新增：一年第幾週（週期）
    "week_cos",            # ✅ 新增：一年第幾週（週期）
    pos_col,
    neg_col,
]
safe_feature_cols = [c for c in safe_feature_cols if c is not None and c in weekly.columns]

print("\n🧱 使用的特徵欄位：", safe_feature_cols)

weekly[safe_feature_cols] = weekly[safe_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# （可選）檢查週期特徵長相
print("\n🗓 week feature sample:")
print(weekly[["start_date", "week_of_year", "week_sin", "week_cos"]].head(10))

# ========= 6. 建序列（嚴格連續週：每次差 7 天） =========
X_seq, y_seq_int, movie_ids = [], [], []
skipped_non_consecutive = 0
debug_examples = []

for movie_id, g in weekly.groupby("movie_id"):
    g = g.sort_values("start_date").reset_index(drop=True)

    dates = g["start_date"].to_numpy()
    vals  = g[safe_feature_cols].to_numpy().astype(np.float32)
    labs  = g["y_class"].to_numpy().astype(int)

    if len(g) <= SEQ_LEN:
        continue

    for t in range(SEQ_LEN, len(g)):
        win_dates = dates[t-SEQ_LEN:t+1]  # [t-2, t-1, t]

        if not (win_dates[-1] > win_dates[-2]):
            raise RuntimeError(f"排序異常：{movie_id} 目標週日期不大於前一週。")

        diffs = np.diff(win_dates).astype("timedelta64[D]").astype(int)
        if not np.all(diffs == 7):
            skipped_non_consecutive += 1
            continue

        X_seq.append(vals[t-SEQ_LEN:t, :])   # (2, n_features)
        y_seq_int.append(labs[t])
        movie_ids.append(movie_id)

        if len(debug_examples) < DEBUG_SHOW_EXAMPLES:
            debug_examples.append((movie_id, list(dates[t-SEQ_LEN:t]), dates[t]))

X_seq = np.array(X_seq, dtype=np.float32)
y_seq_int = np.array(y_seq_int, dtype=int)
movie_ids = np.array(movie_ids)

print("\n✅ LSTM 序列資料形狀：", X_seq.shape)
print("⛔ 因非連續週被略過的窗數：", skipped_non_consecutive)

if len(X_seq) == 0:
    raise RuntimeError("⚠️ 沒有產生任何序列樣本（嚴格連續週條件太嚴）。")

print("\n🔎 範例檢查：前兩週 -> 第三週（日期必定 +7, +7）")
for m, win_ds, tgt_d in debug_examples:
    print("movie:", m,
          " | win:", [str(d)[:10] for d in win_ds],
          " -> target:", str(tgt_d)[:10])

# 檢查 NaN / inf
print("\n🔍 檢查 X_seq 是否有 NaN / inf")
print("NaN:", np.isnan(X_seq).any(), " inf:", np.isinf(X_seq).any())
mask_bad = ~np.isfinite(X_seq)
if mask_bad.any():
    print("⚠️ 發現非有限值，將其設為 0")
    X_seq[mask_bad] = 0.0

# ========= 7. GroupShuffleSplit 依電影切 train/test =========
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_seq, y_seq_int, groups=movie_ids))

X_train_raw, X_test_raw = X_seq[train_idx], X_seq[test_idx]
y_train_int, y_test_int = y_seq_int[train_idx], y_seq_int[test_idx]

train_movies = np.unique(movie_ids[train_idx])
test_movies  = np.unique(movie_ids[test_idx])
overlap = set(train_movies) & set(test_movies)

print("\n🎞 訓練電影數：", len(train_movies))
print("🎞 測試電影數：", len(test_movies))
print("🎯 訓練 / 測試電影交集：", overlap)
if len(overlap) != 0:
    raise RuntimeError("切分失敗：訓練/測試電影有重疊！")

# ✅ 印出 train/test 類別分布
print("\n📊 Train class counts:", np.bincount(y_train_int, minlength=3))
print("📊 Test  class counts:", np.bincount(y_test_int,  minlength=3))

# ========= 8. 標準化（只用訓練集 fit） =========
n_features = X_train_raw.shape[2]
scaler = StandardScaler()
scaler.fit(X_train_raw.reshape(-1, n_features))

X_train = scaler.transform(X_train_raw.reshape(-1, n_features)).reshape(X_train_raw.shape)
X_test  = scaler.transform(X_test_raw.reshape(-1, n_features)).reshape(X_test_raw.shape)

# one-hot labels
y_train_cat = to_categorical(y_train_int, num_classes=3)
y_test_cat  = to_categorical(y_test_int, num_classes=3)

# ========= 9. 建立 BiLSTM 模型 =========
seq_len = X_train.shape[1]

model = Sequential([
    tf.keras.Input(shape=(seq_len, n_features)),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax"),
])

optimizer = tf.keras.optimizers.Adam(
    learning_rate=3e-4,
    clipnorm=1.0,
)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)